# OYO DataPortal Scraper Engine

This notebook runs the fully automated headless scraper pipeline. **No sudo, no apt-get, no Xvfb required.** It uses Chrome's native `--headless=new` engine.

**Step 1:** Run the cell below to set up your environment (only needed once).

In [ ]:
import subprocess
import sys

try:
    subprocess.run(["conda", "env", "create", "-f", "environment.yml"], check=True)
except subprocess.CalledProcessError:
    print('Env already exists or creation failed')

subprocess.run([sys.executable, "-m", "pip", "install", "playwright", "pandas", "requests", "beautifulsoup4", "lxml"], check=True)
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)

**Step 2:** Inject your task directly into the pipeline queue.

In [ ]:
import json, uuid, os
from datetime import datetime

os.makedirs('admin', exist_ok=True)

# Load existing tasks if any
tasks_file = 'admin/tasks.json'
if os.path.exists(tasks_file):
    with open(tasks_file, 'r') as f:
        try:
            tasks = json.load(f)
        except:
            tasks = []
else:
    tasks = []

# Inject new task
tasks.append({
    "id": uuid.uuid4().hex[:8], 
    "created_at": datetime.now().isoformat(),
    "initial_wording": "groupbookings_", 
    "search_terms": ["Trucking Company"],
    "locality": "Texas", 
    "zip_codes": ["78045", "77073"], 
    "mode": "emails",
    "completed_zips": {}, 
    "status": "pending"
})

with open(tasks_file, 'w') as f:
    json.dump(tasks, f, indent=4)
    
print("\n[+] Setup complete! Task injected.")

**Step 3:** Start the Pipeline Engine!

In [ ]:
import os
import subprocess
import sys

os.makedirs('golden_profile', exist_ok=True)
subprocess.run([sys.executable, "server_pipeline.py", "--golden-dir", "./golden_profile"])